In [1]:
import os
from pykalshi import KalshiClient
import pandas as pd
from pykalshi import CandlestickPeriod
from datetime import datetime, timezone

## set key id and path to private key
os.environ["KALSHI_API_KEY_ID"] = "2ac21d6d-0110-4df1-b1c1-7eb77d0a71c1"
os.environ["KALSHI_PRIVATE_KEY_PATH"] = "../kalshi_private_key.pem"


In [2]:
### Extract list of tickers asssociated w fed decision market
client = KalshiClient()
markets = client.get_markets(series_ticker="KXFEDDECISION")
df = markets.to_dataframe()
tickers = df["ticker"].tolist()
#print(df[["ticker", "event_ticker", "title"]])

df.to_csv("kalshi_markets.csv", index=False)
#print(df.columns)

In [3]:
### extract historical data for each ticker and save to csv
full_df_list = []  

## set current timestamp to use in case close data is in future
current_ts = int(datetime.now(timezone.utc).timestamp())

## loop though tickers
for ticker in tickers:
    try:
        ticker_data = df[df["ticker"] == ticker]
        if ticker_data.empty:
            print(f"Warning: No data found for {ticker}")
            continue
        
        #get start and end timestamps from market data
        start_ts = int(datetime.fromisoformat(
            ticker_data["open_time"].iloc[0].replace("Z", "+00:00")
        ).timestamp())
        end_ts = int(datetime.fromisoformat(
            ticker_data["close_time"].iloc[0].replace("Z", "+00:00")
        ).timestamp())

        # use todays date for timestamp if close time is in the future
        if end_ts > current_ts:
            end_ts = current_ts
        
        #pull data
        market = client.get_market(ticker)
        candles = market.get_candlesticks(start_ts, end_ts, period=CandlestickPeriod.ONE_HOUR)
        sample = candles.to_dataframe()
        full_df_list.append(sample)
        
    except Exception as e:
        print(f"Error processing {ticker}: {e}")
        continue

# Concatenate and save
full_df = pd.concat(full_df_list, ignore_index=True)
full_df.to_csv("kalshi_full_data.csv", index=False)